In [29]:
import matplotlib.pyplot as plt
import seaborn as sns
from sdpm.loader import load_model_from_config
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sdpm.util import np2torch
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
import pandas as pd
import json

In [35]:
DEVICE = "cuda:1"
SEED = 42
TIMES_N = 512
df_all_results = pd.read_csv("sdpm/experiments/results/final_results.csv", sep=";")

In [36]:
def get_run_params(ds_name):
    config_row = df_all_results.query(f"ds_name == '{ds_name}' and model_name == 'sdpm_mlp'")
    params = config_row['params'].iloc[0]
    fold_seed = config_row['fold_seed'].iloc[0]
    return json.loads(params), int(fold_seed)

def get_ds(ds_name, seed):
    skf = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )
    data_path = f'data/{ds_name}.npz'
    data = np.load(data_path)
    y = data['y']
    X_num = data['X_num']
    X_cat = data['X_cat']
    train_idx, test_idx = next(skf.split(X_num, y["event"]))
    X_num_train, X_cat_train, y_train = X_num[train_idx], X_cat[train_idx], y[train_idx]
    X_num_test, X_cat_test, y_test = X_num[test_idx], X_cat[test_idx], y[test_idx]
    train_idx, val_idx = train_test_split(
        np.arange(y_train.shape[0]),
        test_size=0.25,
        stratify=y_train["event"],
        random_state=seed,
    )

    X_num_ptrain, X_num_pval = X_num_train[train_idx], X_num_train[val_idx]
    X_cat_ptrain, X_cat_pval = X_cat_train[train_idx], X_cat_train[val_idx]
    y_ptrain, y_pval = y_train[train_idx], y_train[val_idx]
    
    data_splits = {
        'X_num_train': X_num_ptrain,
        'X_cat_train': X_cat_ptrain,
        'y_train': y_ptrain,
        'X_num_val': X_num_pval,
        'X_cat_val': X_cat_pval,
        'y_val': y_pval,
        'X_num_test': X_num_test,
        'X_cat_test': X_cat_test,
        'y_test': y_test,
    }
    data.close()
    return data_splits

def train_sdpm(data, config, model_name, seed):
    config['val_metric'] = 'ibs'
    model = load_model_from_config(
        model_name=model_name,
        config=config,
        device=DEVICE,
        X_num_train=data['X_num_train'],
        X_cat_train=data['X_cat_train'],
        y_train=data['y_train'],
        verbose=0,
        seed=seed
    )
    model.fit(data['X_num_train'], data['X_cat_train'], data['y_train'], (data['X_num_val'], data['X_cat_val'], data['y_val']))
    return model

def raw_predict(sdpm, x_num: np.ndarray,
                x_cat: np.ndarray | None, 
                times_n, batch_size = 32768,
                seed: int | None = SEED):
    with torch.no_grad():
        batch_size = times_n * max(1, batch_size // times_n)
        X_num = np2torch(x_num, device=DEVICE)
        X_cat = np2torch(x_cat, device=DEVICE, dtype=torch.long) if x_cat is not None else None
        get_ds = lambda x_num, x_cat=None: TensorDataset(x_num) if x_cat is None else TensorDataset(x_num, x_cat)
        ds_all = get_ds(X_num, X_cat)
        dl_all = DataLoader(ds_all, batch_size=128, shuffle=False)
        pred_raw_time_list = []
        pred_raw_c_list = []
        for data_outer in dl_all:
            repeater = lambda t: t[:, None, :].repeat(1, times_n, 1).flatten(end_dim=1)
            data_outer = map(repeater, data_outer)
            ds_inner = get_ds(*data_outer)
            dl_inner = DataLoader(ds_inner, batch_size=batch_size, shuffle=False)
            for inner_batch in dl_inner:
                x_num_cur = inner_batch[0]
                x_cat_cur = inner_batch[1] if len(inner_batch) > 1 else None
                pred_raw = sdpm.diffusion.reverse_process(
                    x_num=x_num_cur,
                    x_cat=x_cat_cur,
                    times_n=times_n,
                    mlp=sdpm.baseline,
                    seed=seed,
                )
                pred_raw_time_list.append(pred_raw[..., 0])
                pred_raw_c_list.append(pred_raw[..., 1])
        t_raw_all = torch.cat(pred_raw_time_list, dim=0).reshape(-1, times_n)
        c_raw_all = torch.cat(pred_raw_c_list, dim=0).reshape(-1, times_n)
        return t_raw_all.cpu().numpy(), c_raw_all.cpu().numpy()

In [37]:
def do_experiment(ds_name):
    print(f'=== {ds_name} ===')
    config, ds_seed = get_run_params(ds_name)
    data = get_ds(ds_name, ds_seed)
    model = train_sdpm(data, config, 'sdpm_mlp', ds_seed)
    model_pure = train_sdpm(data, config, 'sdpm_mlp_pure_all', ds_seed)
    results = {}

    def draw_history(model, postfix):
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].plot(model.loss_hist)
        axes[0].set_title('Loss ' + postfix)
        axes[1].plot(model.val_hist['c_index'], label='C-index')
        axes[1].plot(model.val_hist['auc'], label='AUC')
        axes[1].set_title('Validation ' + postfix)
        axes[1].legend()
        axes[2].plot(model.val_hist['ibs'], label='IBS')
        axes[2].legend()

    draw_history(model, 'with upgrades')
    draw_history(model_pure, 'with NO upgrades')

    def print_score(model):
        score = model.score(data['X_num_test'], data['X_cat_test'], data['y_test'], times_n=TIMES_N, batch_size=32768)
        print(f'C-index: {score[0]}, IBS: {score[1]}, AUC: {score[2]}')
        return score

    print('Scores for model with upgrades:')
    model_score = print_score(model)
    results['c_index_log'] = model_score[0]
    results['ibs_log'] = model_score[1]
    results['auc_log'] = model_score[2]
    
    print('Scores for model with NO upgrades:')
    model_pure_score = print_score(model_pure)
    results['c_index_pure'] = model_pure_score[0]
    results['ibs_pure'] = model_pure_score[1]
    results['auc_pure'] = model_pure_score[2]
    
    t_raw, delta_raw = raw_predict(model, data['X_num_test'], data['X_cat_test'], times_n=TIMES_N, seed=ds_seed)
    t_raw_pure, delta_raw_pure = raw_predict(model_pure, data['X_num_test'], data['X_cat_test'], times_n=TIMES_N, seed=ds_seed)

    up_limit = np.max(data['y_train']['time']) * 2
    t = np.exp(t_raw * model.tau_sigma.item() + model.tau_mean.item())
    assert t.shape == t_raw.shape
    t_pure = t_raw_pure * model_pure.tau_sigma.item() + model_pure.tau_mean.item()
    assert t_pure.shape == t_raw_pure.shape

    results["t_outlier_low_log"] = np.mean(t < 0)
    results["t_outlier_high_log"] = np.mean(t > up_limit)
    results["t_outlier_low_pure"] = np.mean(t_pure < 0)
    results["t_outlier_high_pure"] = np.mean(t_pure > up_limit)

    
    print("Percent of the T < 0 for model with upgrades: ", results["t_outlier_low_log"])
    print("Percent of the T > 2*T_max for model with upgrades: ", results["t_outlier_high_log"])
    
    
    print("Percent of the T < 0 for model with NO upgrades: ", results["t_outlier_low_pure"])
    print("Percent of the T > 2*T_max for model with NO upgrades: ", results["t_outlier_high_pure"])

    
    results['event_rate'] = np.mean(data['y_train']['event'])
    results['event_rate_log'] = np.mean(delta_raw > 0)
    results['event_rate_pure'] = np.mean(delta_raw_pure > 0)
    print(f"Event rate in the {ds_name}:", results['event_rate'])
    print(f"Event rate for model with upgrades:", results['event_rate_log'])
    print(f"Event rate for model with NO upgrades:", results['event_rate_pure'])

    clip = (-5, 5)

    fig, ax = plt.subplots()
    sns.kdeplot(delta_raw.ravel(), bw_adjust=0.01, clip=clip)
    ax.set_title(f"{ds_name}, delta with upgrades")

    fig, ax = plt.subplots()
    sns.kdeplot(delta_raw_pure.ravel(), bw_adjust=0.01, clip=clip)
    ax.set_title(f"{ds_name}, delta with NO upgrades")

    clip = (0, up_limit)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    t_event = t.ravel()[delta_raw.ravel() > 0]
    t_event = t_event[t_event < up_limit]
    sns.kdeplot(t_event, bw_adjust=0.5, clip=clip, ax=axes[0], label='Estimation', color='b', fill=True)
    sns.kdeplot(data['y_test']['time'][data['y_test']['event']], bw_adjust=0.5, clip=clip, ax=axes[0], label='Ground truth', color='r', fill=True)
    axes[0].set_title('Event')
    axes[0].legend()
    
    t_cens = t.ravel()[delta_raw.ravel() <= 0]
    t_cens = t_cens[t_cens < up_limit]
    sns.kdeplot(t_cens, bw_adjust=0.5, clip=clip, ax=axes[1], label='Estimation', color='b', fill=True)
    sns.kdeplot(data['y_test']['time'][~data['y_test']['event']], bw_adjust=0.5, clip=clip, ax=axes[1], label='Ground truth', color='r', fill=True)
    axes[1].set_title('Censor')
    axes[1].legend()
    fig.suptitle(f"{ds_name}, time with upgrades")

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    t_event = t_pure.ravel()[delta_raw.ravel() > 0]
    sns.kdeplot(t_event, bw_adjust=0.5, clip=clip, ax=axes[0], label='Estimation', color='b', fill=True)
    sns.kdeplot(data['y_test']['time'][data['y_test']['event']], bw_adjust=0.5, clip=clip, ax=axes[0], label='Ground truth', color='r', fill=True)
    axes[0].set_title('Event')
    axes[0].legend()
    
    t_cens = t_pure.ravel()[delta_raw.ravel() <= 0]
    sns.kdeplot(t_cens, bw_adjust=0.5, clip=clip, ax=axes[1], label='Estimation', color='b', fill=True)
    sns.kdeplot(data['y_test']['time'][~data['y_test']['event']], bw_adjust=0.5, clip=clip, ax=axes[1], label='Ground truth', color='r', fill=True)
    axes[1].set_title('Censor')
    axes[1].legend()
    fig.suptitle(f"{ds_name}, time with NO upgrades")
    
    return results

In [38]:
ds_list = reversed(['flchain', 'ovarian', 'pbc', 'retinopathy', 'rotterdam', 'seer', 'support', 'tcga_gbm', 'vlbw', 'whas500'])
all_results = {}

In [ ]:
for ds in ds_list:
    all_results[ds] = do_experiment(ds)

In [ ]:
np.savez('log_delta_ablation.npz', all_results)

In [ ]:
df = pd.DataFrame(all_results).T
df = df[sorted(df.columns)]
df

,auc_log,auc_pure,c_index_log,c_index_pure,event_rate,event_rate_log,event_rate_pure,ibs_log,ibs_pure,t_outlier_high_log,t_outlier_high_pure,t_outlier_low_log,t_outlier_low_pure
whas500,0.781570,0.780112,0.755342,0.760470,0.430605,0.466172,0.575016,0.162652,0.167675,0.003563,0.006938,0.0,0.157656
vlbw,0.909192,0.907196,0.878091,0.876698,0.173410,0.146648,0.128024,0.087626,0.080172,0.000441,0.000000,0.0,0.240323
tcga_gbm,0.929159,0.920239,0.890829,0.885193,0.823881,0.846345,0.815869,0.055375,0.048887,0.002006,0.010408,0.0,0.022756
support,0.921822,0.931312,0.893577,0.859648,0.681117,0.698649,0.751417,0.102171,0.092391,0.000437,0.000000,0.0,0.282438
seer,0.735804,0.741681,0.753065,0.751993,0.152894,0.151734,0.158102,0.077059,0.077519,0.000000,0.000000,0.0,0.000421
rotterdam,0.799617,0.802636,0.724398,0.709518,0.426357,0.469509,0.415416,0.144177,0.144224,0.000000,0.000000,0.0,0.070045
retinopathy,0.635924,0.628093,0.571429,0.565169,0.393665,0.442018,0.479463,0.204421,0.199542,0.130346,0.160156,0.0,0.258523
pbc,0.999921,0.999616,0.996804,0.996094,0.384615,0.413170,0.399591,0.010086,0.007352,0.000949,0.007440,0.0,0.018341
ovarian,0.698414,0.693830,0.635017,0.640229,0.596491,0.870391,0.882341,0.143740,0.143451,0.001413,0.000000,0.0,0.073799
flchain,0.951786,0.940189,0.936088,0.935581,0.275519,0.272579,0.274642,0.045130,0.045350,0.000003,0.000000,0.0,0.001085
